<a href="https://colab.research.google.com/github/Coding-7allhh/YT_Auto/blob/main/Part_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# ADVANCED SOCIAL MEDIA METADATA ENGINE (LOCAL RUNTIME ONLY)
# OPENROUTER FREE MODELS | GROQ FALLBACK | GEMINI FALLBACK | 4X CANDIDATES & RANKING
# LOCAL VIDEOS -> DISCORD WEBHOOK -> MULTIPLE CATEGORY CSV LOGS & THUMBNAILS
# =============================================================================

!pip -q install -U groq moviepy requests google-generativeai

import os
import re
import sys
import json
import time
import csv
import requests
from pathlib import Path
from datetime import datetime, timedelta
from getpass import getpass
from IPython.display import clear_output

try:
    from moviepy import VideoFileClip
except ImportError:
    from moviepy.editor import VideoFileClip

from groq import Groq
import google.generativeai as genai

# =============================================================================
# 1. CONFIGURATION
# =============================================================================

SOURCE_FOLDER_PATH = "/content/Ready_!/"
LOCAL_THUMBNAIL_FOLDER = "/content/thumbnails/"
LOCAL_CSV_FOLDER = "/content/metadata_files/"

DISCORD_WEBHOOK_URL = "https://discord.com/api/webhooks/1500184991991075007/rwezPR-91qzm_LCqBlzp28fQcx3sLPaN2EvijQIqBgq3k2RGIC-0gEAEtGuEEt9lq3Dm"

POSTING_SCHEDULE = [
    "10:00 AM",
    "02:00 PM",
    "06:00 PM",
    "10:00 PM"
]

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

# Updated active and valid models on OpenRouter (using active free and reliable models)
OPENROUTER_MODELS = [
    "google/gemini-2.5-flash:free",
    "google/gemini-2.5-pro:free",
    "meta-llama/llama-3.3-70b-instruct",
    "deepseek/deepseek-chat",
    "mistralai/mistral-7b-instruct-v0.1"
]

# Updated active and non-decommissioned models on Groq
GROQ_MODELS = [
    "llama-3.3-70b-specdec",
    "llama-3.2-1b-preview",
    "llama-3.2-3b-preview"
]

SHEET_HEADER = [
    "Video Name",
    "Video Path",
    "Status",
    "Time_zone",
    "Video Link",
    "Thumbnail Link",
    "YT Title",
    "YT Description",
    "YT Tags",
    "TikTok Caption",
    "Instagram Caption",
    "FB Caption",
    "Popular Captions",
    "Hashtags (#)",
    "Trending Event",
    "Keywords",
    "Snap Chat",
    "AI Engine",
    "Ranking Used",
    "Winning Score",
    "Candidate Scores",
    "Generated At"
]

os.makedirs(SOURCE_FOLDER_PATH, exist_ok=True)
os.makedirs(LOCAL_THUMBNAIL_FOLDER, exist_ok=True)
os.makedirs(LOCAL_CSV_FOLDER, exist_ok=True)

# =============================================================================
# 2. API KEYS
# =============================================================================

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass("Enter OpenRouter API key: ").strip()

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("Enter Groq API key: ").strip()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass("Enter Gemini API key (for direct fallback): ").strip()

try:
    groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
except Exception as e:
    print(f"Warning initializing Groq client: {e}")
    groq_client = None

try:
    if GEMINI_API_KEY:
        genai.configure(api_key=GEMINI_API_KEY)
except Exception as e:
    print(f"Warning configuring Gemini: {e}")

# =============================================================================
# 3. LOCAL CSV LOGGING HELPERS FOR MULTIPLE FOLDERS / TYPES
# =============================================================================

def get_csv_path_for_category(category_name):
    # Replaces spaces or non-alphanumeric chars for standard file naming
    safe_name = re.sub(r'[^a-zA-Z0-9_\-]', '_', category_name)
    return os.path.join(LOCAL_CSV_FOLDER, f"generated_metadata_{safe_name}.csv")

def init_csv_if_missing(csv_path):
    if not os.path.exists(csv_path):
        with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(SHEET_HEADER)

def get_existing_processed_videos(csv_path):
    init_csv_if_missing(csv_path)
    processed = set()
    try:
        with open(csv_path, mode="r", encoding="utf-8") as f:
            reader = csv.reader(f)
            header = next(reader, None)
            for row in reader:
                if row:
                    processed.add(row[0])
    except Exception as e:
        print(f"Warning reading CSV {csv_path}: {e}")
    return processed

def get_last_post_time_from_csv(csv_path):
    init_csv_if_missing(csv_path)
    try:
        with open(csv_path, mode="r", encoding="utf-8") as f:
            reader = csv.reader(f)
            rows = list(reader)
            if len(rows) > 1:
                for row in reversed(rows[1:]):
                    if len(row) > 3 and row[3]:
                        value = row[3]
                        try:
                            return datetime.strptime(
                                value, "[%Y-%m-%d %I:%M %p]"
                            ).replace(year=datetime.now().year)
                        except:
                            try:
                                return datetime.strptime(
                                    value, "%Y-%m-%d %I:%M %p"
                                )
                            except (ValueError, TypeError):
                                continue
    except Exception as exc:
        print(f"Schedule read warning from CSV {csv_path}: {exc}")
    return None

def get_next_schedule_slot(last_dt):
    schedule_times = sorted(
        datetime.strptime(t, "%I:%M %p").time()
        for t in POSTING_SCHEDULE
    )

    now = datetime.now()

    if last_dt is None:
        last_dt = now - timedelta(minutes=1)

    base_date = max(last_dt.date(), now.date())

    for schedule_time in schedule_times:
        candidate = datetime.combine(base_date, schedule_time)

        if candidate > last_dt and candidate > now:
            return candidate

    next_date = base_date + timedelta(days=1)

    return datetime.combine(
        next_date,
        schedule_times[0]
    )

# =============================================================================
# 4. DISCORD VIDEO UPLOAD HELPER
# =============================================================================

def upload_video_to_discord(video_path):
    """
    Uploads the local video to Discord via the webhook and returns the attachment URL.
    """
    print(f"  Uploading {os.path.basename(video_path)} to Discord...")
    try:
        with open(video_path, 'rb') as f:
            files = {
                'file': (os.path.basename(video_path), f, 'video/mp4')
            }
            payload = {
                'content': f"Uploaded metadata video: {os.path.basename(video_path)}"
            }
            response = requests.post(DISCORD_WEBHOOK_URL, data=payload, files=files, timeout=300)

            if response.ok:
                try:
                    res_json = response.json()
                    attachments = res_json.get('attachments', [])
                    if attachments:
                        discord_url = attachments[0].get('url', 'N/A')
                        print(f"  Successfully uploaded to Discord! Link: {discord_url}")
                        return discord_url
                except Exception as parse_err:
                    print(f"  Could not parse Discord JSON response: {parse_err}")
                    return "Uploaded (Webhook accepted request)"
            else:
                print(f"  Discord webhook failed with status {response.status_code}: {response.text}")
    except Exception as exc:
        print(f"  Error uploading to Discord: {exc}")
    return "N/A"

# =============================================================================
# 5. JSON AND TEXT HELPERS
# =============================================================================

def extract_json(text):
    text = str(text).strip()

    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")

    if start < 0 or end <= start:
        raise ValueError("AI did not return a JSON object")

    return json.loads(text[start:end + 1])

def normalize_metadata(data):
    required_fields = [
        "yt_title",
        "yt_description",
        "yt_tags",
        "tiktok_caption",
        "instagram_caption",
        "fb_caption",
        "popular_captions",
        "hashtags",
        "trending_event",
        "keywords",
        "snapchat_caption"
    ]

    for field in required_fields:
        data.setdefault(field, "")

    for field in required_fields:
        value = data[field]

        if isinstance(value, list):
            value = ", ".join(map(str, value))

        data[field] = str(value).strip()[:49000]

    return data

# =============================================================================
# 6. OPENROUTER GENERATION
# =============================================================================

def openrouter_generate(prompt):
    if not OPENROUTER_API_KEY:
        raise RuntimeError("OpenRouter API key is missing")

    last_exception = None
    for model_candidate in OPENROUTER_MODELS:
        try:
            response = requests.post(
                OPENROUTER_URL,
                headers={
                    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                    "Content-Type": "application/json",
                    "HTTP-Referer": "https://colab.research.google.com",
                    "X-Title": "Social Media Metadata Ranking Engine"
                },
                json={
                    "model": model_candidate,
                    "messages": [
                        {
                            "role": "system",
                            "content": (
                                "You are an expert social-media metadata "
                                "writer. Return valid JSON only. "
                                "Do not fabricate facts or current trends."
                            )
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    "temperature": 0.7,
                    "max_tokens": 1800,
                    "stream": False
                },
                timeout=60
            )

            if response.ok:
                payload = response.json()
                content = payload["choices"][0]["message"]["content"]
                actual_model = payload.get("model", model_candidate)
                return normalize_metadata(extract_json(content)), actual_model
            else:
                raise RuntimeError(f"HTTP {response.status_code}: {response.text[:200]}")
        except Exception as exc:
            last_exception = exc
            print(f"    OpenRouter model {model_candidate} failed: {exc}. Trying next fallback...")
            time.sleep(1.5)

    raise last_exception or RuntimeError("All configured OpenRouter models failed.")

# =============================================================================
# 7. GROQ GENERATION
# =============================================================================

def groq_generate(prompt):
    if not groq_client:
        raise RuntimeError("Groq API key or client is missing")

    last_exception = None
    for model_candidate in GROQ_MODELS:
        try:
            response = groq_client.chat.completions.create(
                model=model_candidate,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert social-media metadata "
                            "writer. Return valid JSON only. "
                            "Do not fabricate facts or current trends."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.7,
                max_tokens=1800,
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content
            return normalize_metadata(extract_json(content)), model_candidate
        except Exception as exc:
            last_exception = exc
            print(f"    Groq model {model_candidate} failed: {exc}. Trying next fallback...")
            time.sleep(1.5)

    raise last_exception or RuntimeError("All configured Groq models failed.")

# =============================================================================
# 7B. DIRECT GEMINI FREE FALLBACK GENERATION
# =============================================================================

def gemini_generate_direct(prompt):
    if not GEMINI_API_KEY:
        raise RuntimeError("Gemini API key is missing")

    try:
        model = genai.GenerativeModel("gemini-2.5-flash")
        response = model.generate_content(
            prompt,
            generation_config=genai.types.GenerationConfig(
                response_mime_type="application/json",
                temperature=0.7
            )
        )
        return normalize_metadata(extract_json(response.text)), "gemini-2.5-flash"
    except Exception as exc:
        raise RuntimeError(f"Direct Gemini API fallback call failed: {exc}")

# =============================================================================
# 8. QUALITY SCORING ENGINE
# =============================================================================

def score_candidate(metadata, video_context):
    score = 0.0

    title = metadata.get("yt_title", "")
    description = metadata.get("yt_description", "")
    tags = metadata.get("yt_tags", "")
    hashtags = metadata.get("hashtags", "")
    keywords = metadata.get("keywords", "")

    if 30 <= len(title) <= 75:
        score += 20
    elif 15 <= len(title) <= 90:
        score += 12
    elif title:
        score += 5

    if len(description) >= 100:
        score += 15
    elif len(description) >= 40:
        score += 10

    if tags:
        score += 10

    hashtag_count = len(re.findall(r"#\w+", hashtags))

    if 3 <= hashtag_count <= 8:
        score += 15
    elif hashtag_count > 0:
        score += 8

    keyword_count = len([
        item for item in re.split(r"[,;\n]", keywords)
        if item.strip()
    ])

    if 4 <= keyword_count <= 12:
        score += 15
    elif keyword_count > 0:
        score += 8

    context_words = set(re.findall(r"\b[a-zA-Z0-9]{4,}\b", video_context.lower()))
    metadata_text = " ".join([title, description, tags, hashtags, keywords]).lower()
    metadata_words = set(re.findall(r"\b[a-zA-Z0-9]{4,}\b", metadata_text))

    if context_words:
        overlap = len(context_words & metadata_words) / len(context_words)
        score += min(20, overlap * 40)

    if not title:
        score -= 40

    words = re.findall(r"\b[a-zA-Z]{4,}\b", (title + " " + description).lower())
    if words:
        repetition = 1 - len(set(words)) / len(words)
        score -= min(15, repetition * 30)

    return round(max(0, min(100, score)), 2)

# =============================================================================
# 9. STRATEGIES AND 4X ENGINE WITH HYBRID FALLBACKS
# =============================================================================

STRATEGIES = [
    (
        "SEARCH_SEO",
        "Focus on accurate search phrases, relevant keywords, clear wording, and discoverability."
    ),
    (
        "CURIOSITY_HOOK",
        "Create a compelling truthful hook that encourages interest without misleading clickbait."
    ),
    (
        "RETENTION",
        "Focus on the audience's reason to watch, emotional connection, clarity, and strong opening wording."
    ),
    (
        "PLATFORM_OPTIMIZATION",
        "Focus on natural language, platform suitability, readability, and relevant hashtags."
    )
]

def build_prompt(video_name, platform, strategy_name, instructions):
    return f"""
Generate metadata for this video.

VIDEO FILE:
{video_name}

TARGET PLATFORM:
{platform}

AUDIENCE:
India and United Kingdom, where relevant to the actual content.
Use the language appropriate for the video.

STRATEGY:
{strategy_name}

INSTRUCTIONS:
{instructions}

RULES:
- Do not invent events, news, trends, facts, or people.
- Do not claim a hashtag is trending without verified trend data.
- Use accurate and relevant keywords.
- Avoid keyword stuffing and repetitive hashtags.
- Write naturally and professionally.
- Use suitable emojis where appropriate.
- Make platform-specific captions distinct.
- If the content is unknown from the filename, do not pretend that you watched the video.
- Trending Event should say "No live trend data verified" unless actual verified trend information was supplied.

Return ONLY valid JSON with these exact keys:
{{
  "yt_title": "YouTube title",
  "yt_description": "YouTube description",
  "yt_tags": "comma-separated tags",
  "tiktok_caption": "TikTok caption",
  "instagram_caption": "Instagram caption",
  "fb_caption": "Facebook caption",
  "popular_captions": "alternative caption ideas",
  "hashtags": "#relevant #hashtags",
  "trending_event": "verified event or no live trend data verified",
  "keywords": "comma-separated keywords",
  "snapchat_caption": "Snapchat caption"
}}
"""

def generate_4x_ranked_metadata(video_name):
    context = f"Video filename: {video_name}"

    print("\n  [Plan A] Running OpenRouter 4x candidates generation and ranking...")
    candidates = []
    openrouter_failed = False
    for number, (strategy, instructions) in enumerate(STRATEGIES, start=1):
        print(f"    OpenRouter candidate {number}/4: {strategy}")
        prompt = build_prompt(video_name, "Multi-platform social media", strategy, instructions)

        candidate = None
        for attempt in range(2):
            try:
                candidate, model_used = openrouter_generate(prompt)
                candidate["strategy"] = strategy
                candidate["model_used"] = model_used
                candidate["score"] = score_candidate(candidate, context)
                break
            except Exception as exc:
                print(f"      Attempt {attempt+1} failed: {exc}")
                time.sleep(1.5)

        if candidate is None:
            print("    OpenRouter 4x generation incomplete/failed. Switching to Groq 4x Fallback.")
            openrouter_failed = True
            break
        candidates.append(candidate)
        time.sleep(0.5)

    if not openrouter_failed and len(candidates) == 4:
        candidates.sort(key=lambda item: item["score"], reverse=True)
        winner = candidates[0]
        scores = " | ".join(f'{item["strategy"]}:{item["score"]}' for item in candidates)
        print(f"    OpenRouter 4x Winner: {winner["strategy"]} (Score: {winner["score"]})")
        return {
            "status": "openrouter_4x_ranked",
            "ranking_used": True,
            "selected": winner,
            "score": winner["score"],
            "candidate_scores": scores,
            "model_used": winner.get("model_used", ""),
            "error": ""
        }

    print("\n  [Plan B] Running Groq Free API 4x candidates generation and ranking fallback...")
    candidates = []
    groq_failed = False
    for number, (strategy, instructions) in enumerate(STRATEGIES, start=1):
        print(f"    Groq candidate {number}/4: {strategy}")
        prompt = build_prompt(video_name, "Multi-platform social media", strategy, instructions)

        candidate = None
        for attempt in range(2):
            try:
                candidate, model_used = groq_generate(prompt)
                candidate["strategy"] = strategy
                candidate["model_used"] = model_used
                candidate["score"] = score_candidate(candidate, context)
                break
            except Exception as exc:
                print(f"      Groq Attempt {attempt+1} failed: {exc}")
                time.sleep(1.5)

        if candidate is None:
            print("    Groq 4x generation incomplete/failed.")
            groq_failed = True
            break
        candidates.append(candidate)
        time.sleep(0.5)

    if not groq_failed and len(candidates) == 4:
        candidates.sort(key=lambda item: item["score"], reverse=True)
        winner = candidates[0]
        scores = " | ".join(f'{item["strategy"]}:{item["score"]}' for item in candidates)
        print(f"    Groq 4x Winner: {winner["strategy"]} (Score: {winner["score"]})")
        return {
            "status": "groq_4x_ranked",
            "ranking_used": True,
            "selected": winner,
            "score": winner["score"],
            "candidate_scores": scores,
            "model_used": winner.get("model_used", ""),
            "error": ""
        }

    print("\n  [Plan C] Both OpenRouter & Groq 4x plans failed. Running single direct Gemini fallback (1x Generation, no ranking)...")
    fallback_prompt = build_prompt(
        video_name,
        "Multi-platform social media",
        "BALANCED_FALLBACK",
        "Generate one single balanced, highly accurate metadata set immediately using your Gemini free model."
    )

    try:
        fallback, model_used = gemini_generate_direct(fallback_prompt)
        return {
            "status": "gemini_single_fallback",
            "ranking_used": False,
            "selected": fallback,
            "score": "",
            "candidate_scores": "",
            "model_used": model_used,
            "error": ""
        }
    except Exception as gemini_exc:
        print(f"    Direct Gemini API Fallback failed: {gemini_exc}")

    print("\n  [Plan D] Fallback options exhausted. Trying general single call backups...")
    try:
        fallback, model_used = groq_generate(fallback_prompt)
        return {
            "status": "groq_single_fallback",
            "ranking_used": False,
            "selected": fallback,
            "score": "",
            "candidate_scores": "",
            "model_used": model_used,
            "error": ""
        }
    except Exception as groq_exc:
         print(f"    Groq single backup failed: {groq_exc}")

    try:
        fallback, model_used = openrouter_generate(fallback_prompt)
        return {
            "status": "openrouter_single_fallback",
            "ranking_used": False,
            "selected": fallback,
            "score": "",
            "candidate_scores": "",
            "model_used": model_used,
            "error": ""
        }
    except Exception as or_exc:
        print(f"    OpenRouter single backup failed: {or_exc}")

    return {
        "status": "failed",
        "ranking_used": False,
        "selected": {},
        "score": "",
        "candidate_scores": "",
        "model_used": "",
        "error": "All model fallback steps failed entirely."
    }

# =============================================================================
# 10. LOCAL THUMBNAIL CREATION
# =============================================================================

def create_thumbnail_locally(video_path, video_name):
    output_filename = f"{Path(video_name).stem}_thumb.jpg"
    destination_path = os.path.join(LOCAL_THUMBNAIL_FOLDER, output_filename)

    try:
        with VideoFileClip(video_path) as clip:
            if clip.duration is None or clip.duration <= 0:
                raise ValueError("Invalid video duration")

            timestamp = min(1.0, max(0, clip.duration / 2))
            clip.save_frame(destination_path, t=timestamp)

        print(f"  Local thumbnail saved to: {destination_path}")
        return destination_path

    except Exception as exc:
        print("  Thumbnail warning:", exc)
        return "N/A"

# =============================================================================
# 11. MAIN PROCESS
# =============================================================================

print("\nStarting Advanced Local Metadata Engine...")
print("Source Folder:", SOURCE_FOLDER_PATH)
print("Local Thumbnails Folder:", LOCAL_THUMBNAIL_FOLDER)

total_processed_count = 0
total_skipped_count = 0
total_failed_count = 0

# Walk through all directories under SOURCE_FOLDER_PATH to separate processes by type (folder)
for root, dirs, files in os.walk(SOURCE_FOLDER_PATH):
    # Only process directories that actually contain video files directly inside them
    videos = sorted([
        name for name in files
        if name.lower().endswith((".mp4", ".mov", ".avi", ".mkv"))
    ])

    if not videos:
        continue

    # Use the leaf folder name as the Category / Type
    category_name = os.path.basename(root) if os.path.basename(root) else "General"
    category_csv_path = get_csv_path_for_category(category_name)

    print(f"\nProcessing Category: '{category_name}'")
    print(f"Target CSV file for this category: {category_csv_path}")

    existing_names = get_existing_processed_videos(category_csv_path)
    last_dt = get_last_post_time_from_csv(category_csv_path)
    rows_to_save = []

    category_processed = 0
    category_skipped = 0
    category_failed = 0

    for video_name in videos:
        if video_name in existing_names:
            print(f"  Skipping existing video in '{category_name}': {video_name}")
            category_skipped += 1
            total_skipped_count += 1
            continue

        video_path = os.path.join(root, video_name)

        next_dt = get_next_schedule_slot(last_dt)
        formatted_time = next_dt.strftime("%Y-%m-%d %I:%M %p")
        last_dt = next_dt

        print("\n  Processing:", video_name)
        print("  Scheduled time:", formatted_time)

        discord_video_link = upload_video_to_discord(video_path)

        try:
            result = generate_4x_ranked_metadata(video_name)

            if result["status"] == "failed":
                raise RuntimeError(result.get("error", "Geneartion failed completely."))

            metadata = result["selected"]

        except Exception as exc:
            print("  Metadata generation failed completely:", exc)
            category_failed += 1
            total_failed_count += 1
            continue

        thumbnail_path = create_thumbnail_locally(video_path, video_name)

        generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

        row = [
            video_name,
            video_path,
            "pending",
            formatted_time,
            discord_video_link,
            thumbnail_path,
            metadata.get("yt_title", ""),
            metadata.get("yt_description", ""),
            metadata.get("yt_tags", ""),
            metadata.get("tiktok_caption", ""),
            metadata.get("instagram_caption", ""),
            metadata.get("fb_caption", ""),
            metadata.get("popular_captions", ""),
            metadata.get("hashtags", ""),
            metadata.get("trending_event", ""),
            metadata.get("keywords", ""),
            metadata.get("snapchat_caption", ""),
            result.get("model_used", ""),
            str(result.get("ranking_used", False)),
            str(result.get("score", "")),
            result.get("candidate_scores", ""),
            generated_at
        ]

        rows_to_save.append(row)
        existing_names.add(video_name)
        category_processed += 1
        total_processed_count += 1

        print("  AI status:", result["status"])
        print("  Ranking used:", result["ranking_used"])

    if rows_to_save:
        try:
            with open(category_csv_path, mode="a", newline="", encoding="utf-8") as csv_file:
                writer = csv.writer(csv_file)
                writer.writerows(rows_to_save)
            print(f"\nSuccessfully appended {len(rows_to_save)} rows to category CSV: {category_csv_path}")
        except Exception as csv_err:
            print(f"\nError writing to category CSV {category_csv_path}: {csv_err}")

print("\n" + "=" * 65)
print("OVERALL PROCESS COMPLETED")
print("=" * 65)
print("Total successfully processed:", total_processed_count)
print("Total skipped existing videos:", total_skipped_count)
print("Total failed videos:", total_failed_count)
print("Local thumbnails directory:", LOCAL_THUMBNAIL_FOLDER)
print("All metadata categories are logged in separate CSV files in local storage.")

Enter OpenRouter API key: ··········
Enter Groq API key: ··········
Enter Gemini API key (for direct fallback): ··········

Starting Advanced Local Metadata Engine...
Source Folder: /content/Ready_!/
Local Thumbnails Folder: /content/thumbnails/

Processing Category: 'Funny'
Target CSV file for this category: /content/metadata_logs/generated_metadata_Funny.csv

  Processing: b0effae0326a0300941e6548284711a5_1793893555417.mp4
  Scheduled time: 2026-11-06 06:00 PM
  Uploading b0effae0326a0300941e6548284711a5_1793893555417.mp4 to Discord...
  Successfully uploaded to Discord! Link: https://cdn.discordapp.com/attachments/1500173892818436179/1568306948418314250/b0effae0326a0300941e6548284711a5_1793893555417.mp4?ex=6aef572a&is=6aee05aa&hm=33594151fbcda8cfddb9625c98a97a562697159a981735afde8f87743cc09767&

  [Plan A] Running OpenRouter 4x candidates generation and ranking...
    OpenRouter candidate 1/4: SEARCH_SEO
    OpenRouter model google/gemini-2.5-flash:free failed: HTTP 404: {"error":{"